In [1]:
import pandas as pd
import os
import csv
import re
import unicodedata
from collections import Counter
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.trainers import BpeTrainer
from tokenizers.processors import TemplateProcessing
from tokenizers import Tokenizer
import torch
import torch.nn as nn
from torch.nn import functional as F
import numpy as np
from sklearn.model_selection import train_test_split

import math

torch.manual_seed(42)

In [2]:
# data = pd.read_csv(r'../data/processed/final_cleaned_jokes.csv')

data = pd.read_csv(
    "/content/sample_data/final_cleaned_jokes.csv",
    engine="python"
)

data = data[data.joke_cleaned.notnull()]

In [3]:
# tokenizer = Tokenizer.from_file(
#     r'../data/processed/tokenizer.json'
# )

# unigram_tokenizer = Tokenizer.from_file(
#     r'../data/processed/tokenizer_unigram.json'
# )

# tokenizer = Tokenizer.from_file(
#      r'/content/sample_data/tokenizer.json'
#  )
tokenizer = Tokenizer.from_file(
     r'/content/sample_data/tokenizer_unigram.json'
 )

In [4]:
data

,rid,stable_id,joke,topic,joke_cleaned,word_count
0,1,0000066daaacde54c609a69be4f00c3336480137,Whoever said imitation is the sincerest form o...,"imitation, flattery, minute",whoever said imitation is the sincerest form o...,23
1,2,00000a79878b3729adc0e47a34dbf5d5484214c0,"You're so fat, they oughta call your dick gary...","oldman, dick, gary","you're so fat , they oughta call your dick gar...",20
2,3,000054bc76448f9b302e6266a72324ecb81d4083,I couldn't sleep last night. I was wondering w...,"night, sun",i couldn't sleep last night . i was wondering ...,18
3,4,0000b76ab05f5d5cf9952d109c5ce7b143ae016f,What's 11 & 2? The Cowboys,cowboys,what's 11 2 ? the cowboys,6
4,5,00015c7571451cc5eb99c24dd7bc46a5ce46b9a1,I just got done apologizing to my barbershop q...,"barbershop, quartet, bucket",i just got done apologizing to my barbershop q...,34
...,...,...,...,...,...,...
726782,726783,ffff42e7a4d6a0ba5d6d548400c2786eca9609c4,There were three dinosaurs who found a magic l...,"dinosaur, shower, genie",there were three dinosaurs who found a magic l...,70
726783,726784,ffff4cf4bb9b9e63636efbb0692d20ea00257b64,What was the hardest part about Michael Jackso...,"michael, jackson, autograph",what was the hardest part about michael jackso...,18
726784,726785,ffff621a689a09dd38aef56fc93b58fba9efd948,My dick is called life. Life is hard,"life, dick",my dick is called life . life is hard,9
726785,726786,ffff6e0d16c8de3d9c1c9a1100747ab55f8d6e5f,What was Jeffrey Epstein humming before dying?...,"republic, jeffrey, epstein",what was jeffrey epstein humming before dying ...,17


In [5]:
# encoded = tokenizer.encode(f"tell me a joke about {data.topic[0]}",data.joke_cleaned[0])
# print(encoded.tokens)
# print(encoded.ids)
# print(tokenizer.decode(encoded.ids, skip_special_tokens=False))
print("----- Unigram Tokenizer -----")
encoded_unigram = tokenizer.encode(f"tell me a joke about {data.topic[0]}",data.joke_cleaned[0])
print(encoded_unigram.tokens)
print(encoded_unigram.ids)
print(tokenizer.decode(encoded_unigram.ids, skip_special_tokens=False))

----- Unigram Tokenizer -----
['[S]', 'Ġtell', 'Ġme', 'Ġ', 'a', 'Ġjoke', 'Ġabout', 'Ġimitation', ',', 'Ġflatter', 'y', ',', 'Ġminute', '[JOKE]', 'Ġwho', 'ever', 'Ġsaid', 'Ġimitation', 'Ġis', 'Ġthe', 'Ġsincere', 's', 't', 'Ġfor', 'm', 'Ġof', 'Ġflatter', 'y', 'Ġhasn', "'", 't', 'Ġhad', 'Ġ', 'a', 'Ġ7', 'y', 'o', 'Ġmimick', 'ing', 'Ġtheir', 'Ġevery', 'Ġword', 'Ġfor', 'Ġthe', 'Ġlast', 'Ġ10', 'Ġminutes', 'Ġ', '.', '[/S]']
[0, 153, 37, 7, 8, 115, 74, 11938, 12, 5455, 40, 12, 877, 6, 82, 628, 69, 11938, 27, 11, 5170, 10, 20, 38, 43, 23, 5455, 40, 1451, 14, 20, 90, 7, 8, 494, 40, 85, 24566, 30, 113, 202, 408, 38, 11, 168, 282, 369, 7, 9, 1]
[S] tell me a joke about imitation, flattery, minute[JOKE] whoever said imitation is the sincerest form of flattery hasn't had a 7yo mimicking their every word for the last 10 minutes .[/S]


In [6]:
sequences = [f"tell me a joke about {t} [JOKE] {j}"
             for t,j in zip(data.topic, data.joke_cleaned)]

In [ ]:
# tokenized = tokenizer.encode_batch(sequences)

In [ ]:
# tokenized_ids = [x.ids for x in tokenized]

In [ ]:
# tokenized_ids = torch.tensor(tokenized_ids)

In [ ]:
# tokenized_ids.shape

In [ ]:
# #shuffle data
# idx = torch.randperm(len(tokenized_ids))
# tokenized_ids = tokenized_ids[idx]

# data_len   = len(tokenized_ids)
# train_data = tokenized_ids[:int(data_len * 0.9)]
# test_data  = tokenized_ids[int(data_len * 0.9):]

In [ ]:
# train_data

tensor([[  0, 373, 140,  ...,   2,   2,   2],
        [  0, 373, 140,  ...,   2,   2,   2],
        [  0, 373, 140,  ...,   2,   2,   2],
        ...,
        [  0, 373, 140,  ...,   2,   2,   2],
        [  0, 373, 140,  ...,   2,   2,   2],
        [  0, 373, 140,  ...,   2,   2,   2]])

In [7]:
class decoderConfig:
    def __init__(self, vocab_size, block_size,
                 n_layer=4, n_head=4, n_embd=256, dropout=0.1):
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.n_layer = n_layer
        self.n_head = n_head
        self.n_embd = n_embd
        self.dropout = dropout


In [8]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config: decoderConfig):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.n_head = config.n_head
        self.head_dim = config.n_embd // config.n_head

        self.qkv = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.proj = nn.Linear(config.n_embd, config.n_embd)
        self.attn_drop = nn.Dropout(config.dropout)
        self.resid_drop = nn.Dropout(config.dropout)

        mask = torch.tril(torch.ones(config.block_size, config.block_size))
        # shape: (1, 1, T, T) so it can broadcast over batch & heads
        self.register_buffer("mask", mask.view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()
        # B: batch size (number of sequences).
        # T: sequence length (tokens per sequence).
        # C: embedding dimension (n_embd).
        qkv = self.qkv(x)  # (B, T, 3C)
        q, k, v = qkv.split(C, dim=2)

        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)  # (B, nh, T, hs)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)  # (B, nh, T, T)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_drop(att)

        y = att @ v  # (B, nh, T, hs)
        y = y.transpose(1, 2).contiguous().view(B, T, C)  # (B, T, C)
        y = self.resid_drop(self.proj(y))
        return y


class MLP(nn.Module):
    def __init__(self, config: decoderConfig):
        super().__init__()
        self.fc1 = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.fc2 = nn.Linear(4 * config.n_embd, config.n_embd)
        self.act = nn.GELU()
        self.drop = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x


class Block(nn.Module):
    def __init__(self, config: decoderConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.ln2 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


In [9]:
class decoder(nn.Module):
    def __init__(self, config: decoderConfig):
        super().__init__()
        self.config = config

        self.tok_emb = nn.Embedding(config.vocab_size, config.n_embd)
        self.pos_emb = nn.Embedding(config.block_size, config.n_embd)
        self.drop = nn.Dropout(config.dropout)

        self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layer)])
        self.ln_f = nn.LayerNorm(config.n_embd)
        self.head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        # idx: (B, T)
        B, T = idx.size()
        assert T <= self.config.block_size, "Sequence too long for block_size."

        pos = torch.arange(0, T, device=idx.device).unsqueeze(0)  # (1, T)

        x = self.tok_emb(idx) + self.pos_emb(pos)  # (B, T, C)
        x = self.drop(x)

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.head(x)  # (B, T, vocab_size)

        loss = None
        if targets is not None:
            # shift by one
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1)
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        # idx: (B, T)
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config.block_size:]  # crop if too long
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature  # last time step

            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float("inf")

            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)  # (B, 1)
            idx = torch.cat((idx, next_token), dim=1)

        return idx


In [10]:


# Example: data.topic and data.joke_cleaned exist
sequences = [
    f"tell me a joke about {t} [JOKE] {j}"
    for t, j in zip(data.topic, data.joke_cleaned)
]
print(sequences[0])


tell me a joke about imitation, flattery, minute [JOKE] whoever said imitation is the sincerest form of flattery hasn't had a 7yo mimicking their every word for the last 10 minutes .


In [11]:
def chunked(iterable, batch_size):
    for i in range(0, len(iterable), batch_size):
        yield iterable[i:i+batch_size]

tokenized_ids = []

for batch in chunked(sequences, batch_size=1024):   # try 512 / 1024 / 2048
    encoded_batch = tokenizer.encode_batch(batch)
    for e in encoded_batch:
        ids = e.ids
        if len(ids) > 1:
            tokenized_ids.append(ids)

print("Number of sequences:", len(tokenized_ids))
print("Example lengths:", [len(tokenized_ids[i]) for i in range(3)])
print("First sequence token ids:", tokenized_ids[0])


Number of sequences: 726786
Example lengths: [256, 256, 256]
First sequence token ids: [0, 153, 37, 7, 8, 115, 74, 11938, 12, 5455, 40, 12, 877, 7, 6, 82, 628, 69, 11938, 27, 11, 5170, 10, 20, 38, 43, 23, 5455, 40, 1451, 14, 20, 90, 7, 8, 494, 40, 85, 24566, 30, 113, 202, 408, 38, 11, 168, 282, 369, 7, 9, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]


In [12]:
pad_id = tokenizer.token_to_id("[PAD]")
print("PAD id:", pad_id)  # should be 2

clean_tokenized_ids = []

for seq in tokenized_ids:
    # drop all PAD tokens
    ids = [tok for tok in seq if tok != pad_id]
    if len(ids) > 1:
        clean_tokenized_ids.append(ids)

print("Old #sequences:", len(tokenized_ids))
print("New #sequences:", len(clean_tokenized_ids))
print("Example lengths (no PAD):", [len(clean_tokenized_ids[i]) for i in range(3)])


PAD id: 2
Old #sequences: 726786
New #sequences: 726786
Example lengths (no PAD): [51, 47, 38]


In [ ]:
# # Flatten all sequences into one long list
# all_ids = [tok for seq in tokenized_ids for tok in seq]

# data_ids = torch.tensor(all_ids, dtype=torch.long)
# print("Total tokens in corpus:", data_ids.shape[0])
# print("First 20 tokens:", data_ids[:20].tolist())
# print(tokenizer.decode(data_ids[:50].tolist()))


In [ ]:
# n = int(0.9 * len(data_ids))
# train_data = data_ids[:n]
# val_data   = data_ids[n:]

# print("Train tokens:", train_data.shape[0])
# print("Val tokens:",   val_data.shape[0])


In [13]:


# 1) Flatten
all_ids = []
for seq in clean_tokenized_ids:
    all_ids.extend(seq)
data_ids = torch.tensor(all_ids, dtype=torch.long)

print("Total tokens in corpus (no PAD):", data_ids.shape[0])
print("=======================================")
print("First 30 tokens:", data_ids[:30].tolist())
print("=======================================")
print(tokenizer.decode(data_ids[:80].tolist(), skip_special_tokens=False))
print("=======================================")
print(tokenizer.decode(all_ids[:200], skip_special_tokens=False))

# 2) Train/val split
n = int(0.9 * len(data_ids))
train_data = data_ids[:n]
val_data   = data_ids[n:]

print("Train tokens:", train_data.shape[0])
print("Val tokens:",   val_data.shape[0])


Total tokens in corpus (no PAD): 42317125
First 30 tokens: [0, 153, 37, 7, 8, 115, 74, 11938, 12, 5455, 40, 12, 877, 7, 6, 82, 628, 69, 11938, 27, 11, 5170, 10, 20, 38, 43, 23, 5455, 40, 1451]
[S] tell me a joke about imitation, flattery, minute [JOKE] whoever said imitation is the sincerest form of flattery hasn't had a 7yo mimicking their every word for the last 10 minutes .[/S][S] tell me a joke about oldman, dick, gary [JOKE] you're so fat , they oughta call your dick
[S] tell me a joke about imitation, flattery, minute [JOKE] whoever said imitation is the sincerest form of flattery hasn't had a 7yo mimicking their every word for the last 10 minutes .[/S][S] tell me a joke about oldman, dick, gary [JOKE] you're so fat , they oughta call your dick gary oldman  ... cause it always disappears into a roll .[/S][S] tell me a joke about night, sun [JOKE] i couldn't sleep last night . i was wondering where the sun went then it dawned on me[/S][S] tell me a joke about cowboys [JOKE] what's

In [14]:
vocab_size = tokenizer.get_vocab_size()
block_size = 256

config = decoderConfig(
    vocab_size=vocab_size,
    block_size=block_size,
    n_layer=6,
    n_head=6,
    n_embd=384,
    dropout=0.1
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model = decoder(config).to(device)


Using device: cuda


In [15]:

batch_size = 32

def get_batch(split):
    data = train_data if split == "train" else val_data
    # we need at least block_size+1 tokens to make (x,y)
    ix = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i : i + block_size]     for i in ix])
    y = torch.stack([data[i + 1 : i + 1 + block_size] for i in ix])
    # shapes: (B, block_size)
    return x.to(device), y.to(device)

xb, yb = get_batch("train")
print("xb shape:", xb.shape, "yb shape:", yb.shape)

xb shape: torch.Size([32, 256]) yb shape: torch.Size([32, 256])


In [57]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)

max_steps = 20000  # start small, you can increase later
eval_interval = 200

for step in range(max_steps):
    model.train()
    xb, yb = get_batch("train")
    logits, loss = model(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    if step % eval_interval == 0:
        model.eval()
        with torch.no_grad():
            xval, yval = get_batch("val")
            _, val_loss = model(xval, yval)
        print(f"step {step}: train loss {loss.item():.4f}, val loss {val_loss.item():.4f}")


step 0: train loss 2.5027, val loss 2.5390
step 200: train loss 2.6140, val loss 2.5227
step 400: train loss 2.5156, val loss 2.5567
step 600: train loss 2.6341, val loss 2.6675
step 800: train loss 2.5665, val loss 2.5200
step 1000: train loss 2.6044, val loss 2.5557
step 1200: train loss 2.5195, val loss 2.4257
step 1400: train loss 2.5491, val loss 2.4549
step 1600: train loss 2.3521, val loss 2.6069
step 1800: train loss 2.4863, val loss 2.5120
step 2000: train loss 2.5780, val loss 2.5073
step 2200: train loss 2.5200, val loss 2.3990
step 2400: train loss 2.4905, val loss 2.4886
step 2600: train loss 2.4848, val loss 2.5209
step 2800: train loss 2.4402, val loss 2.5859
step 3000: train loss 2.3736, val loss 2.4243
step 3200: train loss 2.4811, val loss 2.3182
step 3400: train loss 2.3762, val loss 2.4511
step 3600: train loss 2.4357, val loss 2.4574
step 3800: train loss 2.4709, val loss 2.4142
step 4000: train loss 2.4990, val loss 2.2910
step 4200: train loss 2.3746, val loss 2.

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [58]:
# ckpt_path = "decoder_jokes_step4800.pt"
# ckpt_path = "/content/drive/MyDrive/decoder_jokes_step4800.pt"
ckpt_path = "/content/drive/MyDrive/decoder_unigram_jokes_step20000.pt"

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "step": step,                     # last training step
        "config": model.config,
        "vocab_size": tokenizer.get_vocab_size(),
    },
    ckpt_path,
)

print("Saved checkpoint to", ckpt_path)


Saved checkpoint to /content/drive/MyDrive/decoder_unigram_jokes_step20000.pt


In [ ]:
# vocab_size = tokenizer.get_vocab_size()
# block_size = 256

# config = decoderConfig(
#     vocab_size=vocab_size,
#     block_size=block_size,
#     n_layer=6,
#     n_head=6,
#     n_embd=384,
#     dropout=0.1
# )

# device = "cuda" if torch.cuda.is_available() else "cpu"
# print("Using device:", device)

# model = decoder(config).to(device)
# optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
# ckpt_path = "/content/drive/MyDrive/decoder_unigram_jokes_step5000.pt"

# checkpoint = torch.load(ckpt_path, map_location=device)

# model.load_state_dict(checkpoint["model_state_dict"])
# optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
# step = checkpoint.get("step", 0)

# print(f"✅ Loaded checkpoint from step {step}")


In [ ]:
print(tokenizer.decode(clean_tokenized_ids[0], skip_special_tokens=False))


[S] tell me a joke about imitation, flattery, minute [JOKE] whoever said imitation is the sincerest form of flattery hasn't had a 7yo mimicking their every word for the last 10 minutes .[/S]


In [73]:
s_id = tokenizer.token_to_id("[S]")
end_id = tokenizer.token_to_id("[/S]")

def generate_joke(topic, max_new_tokens=80, temperature=0.8, top_k=50):
    core = f" tell me a joke about {topic} [JOKE]"
    core_ids = tokenizer.encode(core, add_special_tokens=False).ids
    idx = torch.tensor([[s_id] + core_ids], dtype=torch.long, device=device)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits, _ = model(idx[:, -model.config.block_size:])
            logits = logits[:, -1, :] / temperature

            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float("inf")

            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            next_id = next_token.item()
            idx = torch.cat((idx, next_token), dim=1)

            #  stop at [/S]
            if next_id == end_id:
                break

    out_ids = idx[0].tolist()

    # cut off the generated [/S] if it exists
    if end_id in out_ids:
        out_ids = out_ids[: out_ids.index(end_id)]

    text = tokenizer.decode(out_ids, skip_special_tokens=False)

    # optional: return only the joke part after [JOKE]
    # marker = "[JOKE]"
    # if marker in text:
    #     text = text.split(marker, 1)[1].strip()

    return text


In [74]:
print("----")
print(generate_joke("rain and umbrellas"))


----
[S] tell me a joke about rain and umbrellas [JOKE] i got pulled over by the rain . now i'm in the rain dying .


In [75]:
print(generate_joke("weather"))


[S] tell me a joke about weather [JOKE] i was wondering why the weather was so cold ...  i was wondering why it would take so long to get the uk to turn it on .


In [83]:
print(generate_joke("family"))


[S] tell me a joke about family [JOKE] i hate when people say i like my family ...  i don't want to be cremated , and i don't want to be cremated .


In [97]:
print(generate_joke("sun and sky"))


[S] tell me a joke about sun and sky [JOKE] the sun and the sky are fighting . the sun and the moon are fighting . then it dawned on you .


In [106]:
print(generate_joke("elephant"))


[S] tell me a joke about elephant [JOKE] what do you get when you cross an elephant with a rhino ? poached


In [64]:
import re
import numpy as np

def _extract_joke_region(text, start_marker="[JOKE]", end_marker="[/S]"):
    # cut to the generated joke only
    if start_marker in text:
        text = text.split(start_marker, 1)[1]
    if end_marker in text:
        text = text.split(end_marker, 1)[0]
    # remove any remaining special tokens like [S]
    text = re.sub(r"\[[A-Z\/]+\]", "", text)
    return text.strip()

def _topic_in_text(joke_text, topic):
    j = joke_text.lower()
    # require each topic word to appear as a whole word
    words = re.findall(r"[a-z0-9]+", topic.lower())
    return all(re.search(rf"\b{re.escape(w)}\b", j) for w in words)

def evaluate_topic_inclusion_strict(model, tokenizer, topics,
                                    max_new_tokens=80, temperature=0.8, top_k=50,
                                    show_examples=True):
    hits, total = 0, 0
    results = []
    for topic in topics:
        gen = generate_joke(topic, max_new_tokens=max_new_tokens,
                            temperature=temperature, top_k=top_k)
        joke_only = _extract_joke_region(gen)
        included = _topic_in_text(joke_only, topic)
        hits += int(included); total += 1
        results.append({"topic": topic, "included": included,
                        "full_text": gen, "joke_only": joke_only})
        if show_examples:
            print(f"\n--- Topic: {topic} ---")
            print(f"Included: {included}")
            print(f"Joke-only: {joke_only}")
    rate = hits / max(total, 1)
    print("\n========== SUMMARY ==========")
    print(f"Topics tested: {total}")
    print(f"Included topic in joke: {hits}")
    print(f"Topic inclusion rate: {rate*100:.1f}%")
    return {"inclusion_rate": rate, "total": total, "hits": hits, "results": results}


In [65]:
def _word_tokens(s: str):
    # simple word tokens for diversity / repetition checks
    return re.findall(r"[a-z0-9']+", s.lower())

def evaluate_diversity_from_results(results):
    """
    Distinct-1 and Distinct-2 over the joke-only text.
    """
    all_unigrams, all_bigrams = set(), set()
    total_unigrams, total_bigrams = 0, 0

    for r in results:
        toks = _word_tokens(r["joke_only"])
        total_unigrams += len(toks)
        total_bigrams  += max(0, len(toks)-1)
        all_unigrams.update(toks)
        all_bigrams.update(zip(toks, toks[1:]))

    distinct1 = (len(all_unigrams) / total_unigrams) if total_unigrams else 0.0
    distinct2 = (len(all_bigrams)  / total_bigrams)  if total_bigrams  else 0.0
    return {"distinct1": distinct1, "distinct2": distinct2}





In [66]:
# =========================
# Self-BLEU-2 / Self-BLEU-3
# =========================
from typing import List, Dict
import numpy as np
import re

try:
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
except ImportError:
    !pip -q install nltk
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

_smooth = SmoothingFunction().method3

def _simple_tokenize(s: str) -> List[str]:
    # basic, robust tokenization for BLEU (lowercase + split on non-letters)
    return re.findall(r"[a-z0-9']+|[^\s\w]", s.lower())

def compute_self_bleu(texts: List[str], n: int = 2) -> float:
    """
    Self-BLEU-n: for each text, compute BLEU-n against the rest as references,
    then average. Lower = less mode collapse (more variety).
    """
    assert n in (2,3,4), "n should be 2, 3, or 4"
    if len(texts) < 2:
        return 0.0

    weights_map = {
        2: (0.5, 0.5, 0.0, 0.0),
        3: (1/3, 1/3, 1/3, 0.0),
        4: (0.25, 0.25, 0.25, 0.25),
    }
    weights = weights_map[n]

    toks = [ _simple_tokenize(t) for t in texts ]
    scores = []
    for i in range(len(toks)):
        hyp = toks[i]
        refs = toks[:i] + toks[i+1:]
        if not refs:
            continue
        # NLTK expects list of references (each a list of tokens)
        score = sentence_bleu(refs, hyp, weights=weights, smoothing_function=_smooth)
        scores.append(score)
    return float(np.mean(scores)) if scores else 0.0

# Convenience wrapper to compute both BLEU-2 and BLEU-3
def compute_self_bleu_2_3(texts: List[str]) -> Dict[str, float]:
    return {
        "Self-BLEU-2": compute_self_bleu(texts, n=2),
        "Self-BLEU-3": compute_self_bleu(texts, n=3),
    }



In [67]:
import math
import torch
import torch.nn.functional as F

def perplexity_on_texts(model, tokenizer, texts, block_size=None, max_examples=500):
    """
    Compute perplexity (PPL) of your decoder-only model on a list of strings,
    using sliding windows that keep input/target lengths equal.
    """
    model.eval()
    device = next(model.parameters()).device
    if block_size is None:
        block_size = getattr(model.config, "block_size", 256)

    total_nll = 0.0
    total_tokens = 0

    with torch.no_grad():
        ex_count = 0
        for raw in texts:
            if ex_count >= max_examples:
                break
            if not isinstance(raw, str) or not raw.strip():
                continue

            ids = tokenizer.encode(raw, add_special_tokens=False).ids
            if len(ids) < 2:
                continue

            pos = 0
            # We ensure each chunk has equal input/target length = chunk_len
            while pos + 1 < len(ids):
                chunk_len = min(block_size, (len(ids) - 1) - pos)  # leave room for the shift
                if chunk_len <= 0:
                    break

                x = ids[pos : pos + chunk_len]
                y = ids[pos + 1 : pos + 1 + chunk_len]

                x_t = torch.tensor([x], dtype=torch.long, device=device)
                y_t = torch.tensor([y], dtype=torch.long, device=device)

                logits, _ = model(x_t)                    # (B, T_in, V), T_in == chunk_len
                # (optional) safety slice in case any model returns longer T
                if logits.size(1) != y_t.size(1):
                    logits = logits[:, : y_t.size(1), :]

                loss = F.cross_entropy(
                    logits.reshape(-1, logits.size(-1)),
                    y_t.reshape(-1),
                    reduction="sum"
                )
                total_nll += loss.item()
                total_tokens += y_t.numel()

                pos += chunk_len

            ex_count += 1

    if total_tokens == 0:
        return float("inf")
    return math.exp(total_nll / total_tokens)


In [71]:

topics = ["rain","cats","dentists","robots","finance","marriage","face"]
metrics = evaluate_topic_inclusion_strict(
    model, tokenizer, topics,
    max_new_tokens=80, temperature=0.7, top_k=20, show_examples=True
)

res = metrics["results"]

diversity = evaluate_diversity_from_results(res)
print("Diversity:", diversity)

gens = [r["joke_only"] for r in res]

print(compute_self_bleu_2_3(gens))





--- Topic: rain ---
Included: True
Joke-only: i'm sorry i'm not sure what i'm saying , i am just not giving up rain .

--- Topic: cats ---
Included: True
Joke-only: i used to be " cats and dogs "  ... but i just wasn't able to .

--- Topic: dentists ---
Included: False
Joke-only: i was told my dentist was gay . i wasn't gay , but he told me i was gay .

--- Topic: robots ---
Included: True
Joke-only: how many robots does it take to screw in a light bulb ? none . they just beat it up to something .

--- Topic: finance ---
Included: True
Joke-only: i was going to make a " i'm " joke for " finance ...  but it would've been too much .

--- Topic: marriage ---
Included: True
Joke-only: marriage is like marriage . not everyone gets it

--- Topic: face ---
Included: False
Joke-only: i hate when people say i'm condescending ...  it's not because i'm condescending .

========== SUMMARY ==========
Topics tested: 7
Included topic in joke: 5
Topic inclusion rate: 71.4%
Diversity: {'distinct1': 0.

In [72]:
texts = data["joke"].astype(str).tolist()

ppl = perplexity_on_texts(model, tokenizer, texts, max_examples=10000)  # start with 300 for speed
print("PPL:", ppl)

PPL: 175.28427464977102
